# 02 · Naive Bayes categórico para la clase ICA del día siguiente

**Técnica de la Parte 1:** un clasificador Naive Bayes implementado desde cero en `src/model.py` (numpy, sin librerías de ML).

$$P(y \mid x_1,\dots,x_d) \;\propto\; P(y)\prod_{j=1}^{d} P(x_j \mid y), \qquad \hat P(x_j = v \mid y = c) = \frac{N_{cjv} + \alpha}{N_c + \alpha V_j}$$

- $N_c$: días de entrenamiento con clase $c$; $N_{cjv}$: de esos, cuántos tienen la variable $j$ con valor $v$; $V_j$: número de valores posibles de la variable $j$.
- $\alpha$ es el suavizado de Laplace: evita probabilidades cero para combinaciones nunca vistas.
- Se predice la clase con mayor probabilidad posterior, sumando logaritmos.

**Antes de ejecutar:** `python -m src.download_data`, `python -m src.preprocess`.

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import config, features, ica, preprocess

plt.rcParams.update({"figure.figsize": (10, 3.8), "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})

from src import train
from src.model import NaiveBayesCategorico, matriz_confusion, predecir_persistencia

diario = preprocess.cargar_diario()
tabla = features.construir_tabla(diario).dropna(subset=["y"])
corte = pd.Timestamp(config.TEST_DESDE)
entrenamiento, prueba = tabla[tabla["fecha_objetivo"] < corte], tabla[tabla["fecha_objetivo"] >= corte]
print(f"Entrenamiento: {len(entrenamiento):,} filas, hasta {entrenamiento['fecha_objetivo'].max():%Y-%m-%d}")
print(f"Prueba:        {len(prueba):,} filas, desde {prueba['fecha_objetivo'].min():%Y-%m-%d}")
tabla[["estacion", "fecha", "pm25_hoy", *features.NOMBRES, "y"]].head()

## 1. Variables

Cada variable es categórica y solo usa información disponible el día *t* para predecir el día *t+1*.

In [ ]:
pd.DataFrame([{"variable": k, "valores": ", ".join(v)} for k, v in features.VARIABLES.items()])

## 2. Entrenamiento y comparación con las líneas base

`train.entrenar` elige α validando en el último año del entrenamiento, reentrena con todo el entrenamiento y evalúa una sola vez en el periodo de prueba.

- **Clase mayoritaria:** siempre predice la clase más frecuente.
- **Persistencia:** mañana será igual que hoy.
- «Episodio» = Dañina para grupos sensibles o peor.

In [ ]:
nb_modelo, metricas = train.entrenar(diario)
print(f"α elegido: {metricas['alpha']}")
resultados = pd.DataFrame(metricas["resultados"]).T.drop(columns="matriz_confusion").astype(float)
resultados

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 4.2))
cortas = ["Buena", "Aceptable", "Grupos\nsensibles", "Dañina\no peor"]
for ax, nombre in zip(ejes, ["Persistencia", "Naive Bayes (propio)"]):
    m = np.array(metricas["resultados"][nombre]["matriz_confusion"])
    ax.imshow(m, cmap="Blues")
    for i in range(m.shape[0]):
        for j in range(m.shape[1]):
            ax.text(j, i, m[i, j], ha="center", va="center", color="white" if m[i, j] > m.max() / 2 else "black", fontsize=9)
    ax.set_xticks(range(4)); ax.set_xticklabels(cortas, fontsize=8); ax.set_yticks(range(4)); ax.set_yticklabels(cortas, fontsize=8)
    ax.set_xlabel("Clase predicha"); ax.set_ylabel("Clase real"); ax.set_title(nombre); ax.grid(False)
plt.tight_layout(); plt.show()

## 3. Verificación de la implementación

La implementación propia debe dar exactamente las mismas probabilidades que `CategoricalNB` de scikit-learn con el mismo α. scikit-learn se usa **solo** para esta comprobación.

In [ ]:
from sklearn.naive_bayes import CategoricalNB

X_tr, y_tr, X_te = features.matriz_X(entrenamiento), entrenamiento["y"].to_numpy(int), features.matriz_X(prueba)
referencia = CategoricalNB(alpha=metricas["alpha"], min_categories=features.N_CATEGORIAS).fit(X_tr, y_tr)
propias = nb_modelo.predict_proba(X_te)[:, referencia.classes_]
print(f"Diferencia máxima entre probabilidades: {np.abs(propias - referencia.predict_proba(X_te)).max():.2e}")
print(f"Predicciones idénticas: {(nb_modelo.predict(X_te) == referencia.predict(X_te)).all()}")

## 4. Elección de α y aporte de cada variable

In [ ]:
display(pd.DataFrame(metricas["validacion_alpha"]))
ablacion = pd.DataFrame(metricas["ablacion"]).set_index("sin_variable")
completo = metricas["resultados"]["Naive Bayes (propio)"]
ablacion["cambio_en_f1"] = (ablacion["f1_macro"] - completo["f1_macro"]).round(4)
ablacion.sort_values("cambio_en_f1")

Un cambio negativo en F1 al retirar una variable significa que esa variable ayudaba; uno positivo, que estorbaba (Naive Bayes cuenta dos veces la evidencia de variables muy correlacionadas).

## 5. ¿Qué aprendió? P(clase de mañana | nivel de hoy)

Se muestra la posterior usando solo la variable `nivel_hoy` y la probabilidad previa de cada clase.

In [ ]:
j = features.NOMBRES.index("nivel_hoy")
log_post = nb_modelo.log_prior[:, None] + nb_modelo.log_verosimilitud[j]            # [clase, nivel]
post = np.exp(log_post - log_post.max(axis=0))
post = post / post.sum(axis=0)
pd.DataFrame(post.T, index=[f"hoy {n} µg/m³" for n in features.ETIQUETAS_NIVEL], columns=ica.CLASES).round(3)

## 6. Calibración de la probabilidad de episodio

Si el modelo dice «30 % de probabilidad de episodio», ¿ocurre en el 30 % de los casos? Naive Bayes suele dar probabilidades extremas, y eso importa porque el agente las comunica al usuario.

In [ ]:
p_episodio = nb_modelo.predict_proba(X_te)[:, ica.CLASE_EPISODIO:].sum(axis=1)
real = prueba["y"].to_numpy(int) >= ica.CLASE_EPISODIO
bordes = np.array([0, 0.05, 0.2, 0.4, 0.6, 0.8, 1.0001])
grupo = np.digitize(p_episodio, bordes) - 1
calibracion = pd.DataFrame({"prob_predicha_media": pd.Series(p_episodio).groupby(grupo).mean(),
                            "frecuencia_observada": pd.Series(real).groupby(grupo).mean(),
                            "dias": pd.Series(real).groupby(grupo).size()})
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.plot([0, 1], [0, 1], color="#555", ls="--", lw=1)
ax.plot(calibracion["prob_predicha_media"], calibracion["frecuencia_observada"], "o-", color="#1F4E79")
ax.set_xlabel("Probabilidad de episodio predicha"); ax.set_ylabel("Frecuencia observada")
plt.show()
calibracion.round(3)

## 7. Exactitud por estación

In [ ]:
pd.DataFrame(metricas["por_estacion"]).assign(diferencia=lambda d: (d["naive_bayes"] - d["persistencia"]).round(4)).sort_values("diferencia", ascending=False)

## 8. Limitaciones del modelo

- **Independencia condicional.** Naive Bayes supone que las variables son independientes dada la clase. Aquí no lo son (el nivel de hoy, la clase de ayer y el nivel del valle están muy correlacionados), así que la evidencia se cuenta de más y las probabilidades tienden a ser extremas (sección 6).
- **Sin meteorología.** No se usan lluvia, viento ni altura de la capa de mezcla, que son los factores físicos que disparan o disipan un episodio. El modelo solo ve la inercia del propio PM2.5 y el calendario.
- **Clases desbalanceadas.** Los días de episodio son pocos; por eso se reportan F1-macro y recall de episodio además de la exactitud.
- **Días sin dato.** Solo se predice cuando hay promedio válido hoy y ayer.
- **Festivos.** Se tratan como días laborales.
- **Un solo periodo de prueba.** Un año con condiciones atípicas (por ejemplo, El Niño o incendios regionales) puede no representar a los siguientes.

El modelo es un apoyo analítico con fines académicos y no reemplaza el pronóstico oficial del SIATA.